# Continuous-time range-only localization with WNOA

This example estimates a 3D trajectory from range measurements to known landmarks. The trajectory is represented by positions and velocities at four knot times. A white-noise-on-acceleration (WNOA) prior connects consecutive knots, while interpolation factors incorporate ranges measured between them.

The positions use `AugmentedPoint3` and the measurements use `KnownLandmarkSquaredRangeFactorAugmentedPoint3`, both from `gtsam_unstable`. We first solve the resulting nonlinear least-squares problem with GTSAM's Levenberg–Marquardt optimizer. We then rewrite the same problem as a quadratically constrained quadratic program (QCQP) and solve its semidefinite relaxation, which provides a lower bound on the cost. If a feasible solution reaches that bound to within the solver tolerance, it is a global minimizer of the nonlinear problem.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All Rights Reserved. Authors: Frank Dellaert, et al. (see THANKS for the full author list). See LICENSE for the license information.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam_unstable/slam/doc/RangeOnlyWnoaAugmentedPoint.ipynb)

In [ ]:
try:
    import google.colab
except ImportError:
    pass

In [ ]:
import numpy as np
import plotly.graph_objects as go
import gtsam
import gtsam_unstable as gu

## Range measurements and trajectory

We simulate a smooth trajectory observed by four fixed landmarks. Range measurements are taken at the first knot and at two times inside each subsequent knot interval. The knot positions and velocities are the only unknowns; landmark locations remain fixed.

For a landmark at $L_i$ and measured distance $d_i$, the squared-range factor uses

$$
e_i(p(t)) = \|p(t)-L_i\|^2-d_i^2.
$$

If the raw range has standard deviation $\sigma_r$, first-order uncertainty propagation gives $\sigma_{d^2}\approx 2d_i\sigma_r$. We use this value for the one-dimensional factor noise model, since the residual is expressed in squared meters.

In [ ]:
rng = np.random.default_rng(7)

knot_times = np.array([0.0, 1.0, 2.0, 3.0])
measurement_times = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75])
landmarks = np.array([
    [0.0, 0.0, 0.0],
    [3.0, 0.0, 0.0],
    [0.0, 3.0, 0.0],
    [0.0, 0.0, 3.0],
])
range_sigma = 0.02  # m
q_psd = np.full(3, 0.2)  # WNOA acceleration power spectral density


def true_position(t):
    t = np.asarray(t)
    return np.stack((
        0.6 + 0.35 * t + 0.06 * np.sin(0.9 * t),
        -0.2 + 0.1 * t - 0.07 * (1 - np.cos(0.7 * t)),
        0.7 - 0.15 * t + 0.06 * np.sin(1.1 * t),
    ), axis=-1)


def true_velocity(t):
    t = np.asarray(t)
    return np.stack((
        0.35 + 0.054 * np.cos(0.9 * t),
        0.1 - 0.049 * np.sin(0.7 * t),
        -0.15 + 0.066 * np.cos(1.1 * t),
    ), axis=-1)


distances = np.linalg.norm(
    true_position(measurement_times)[:, None, :] - landmarks[None, :, :],
    axis=2,
)
measured_ranges = distances + rng.normal(0, range_sigma, distances.shape)

## Build the factor graph

Each knot has a position key and a velocity key. `WnoaMotionFactorAugmentedPoint3` constrains consecutive knots using the WNOA prior.

A range at the first knot is connected directly to its position. For each intermediate measurement, `WnoaInterpFactorAugmentedPoint3` wraps the squared-range factor and propagates its constraint to the surrounding knot positions and velocities. There are no optimization variables at the intermediate measurement times.

In [ ]:
states = [
    gtsam.StateData(gtsam.symbol("p", i), gtsam.symbol("v", i), float(t))
    for i, t in enumerate(knot_times)
]

initial = gtsam.Values()
for state, t in zip(states, knot_times):
    position = true_position(t) + np.array([0.15, -0.10, 0.13])
    velocity = true_velocity(t) + np.array([-0.10, 0.08, -0.06])
    gu.insertAugmentedPoint3(state.pose, gu.AugmentedPoint3(position), initial)
    initial.insert(state.velocity, velocity)


def build_graph(keep=lambda index, landmark: True):
    """Build the WNOA graph with the ranges selected by keep(time_index, landmark_index)."""
    graph = gtsam.NonlinearFactorGraph()
    for left, right in zip(states[:-1], states[1:]):
        graph.add(gu.WnoaMotionFactorAugmentedPoint3(left, right, q_psd))

    for index, (t, ranges) in enumerate(zip(measurement_times, measured_ranges)):
        # Ranges at the first knot connect directly to its position; the others are interpolated.
        interpolated_state = gtsam.StateData(
            gtsam.symbol("p", 100 + index), gtsam.symbol("v", 100 + index), float(t))
        position_key = states[0].pose if index == 0 else interpolated_state.pose
        for j, (landmark, distance) in enumerate(zip(landmarks, ranges)):
            if not keep(index, j):
                continue
            noise = gtsam.noiseModel.Isotropic.Sigma(1, 2 * distance * range_sigma)
            range_factor = gu.KnownLandmarkSquaredRangeFactorAugmentedPoint3(
                position_key, landmark, float(distance), noise)
            graph.add(range_factor if index == 0 else gu.WnoaInterpFactorAugmentedPoint3(
                range_factor, set(states), {interpolated_state}, q_psd, True, True))
    return graph


graph = build_graph()
print(f"Knot states: {len(states)}")
print(f"Range measurements: {measured_ranges.size}")
print(f"Factor graph: {graph.size()} factors, {initial.size()} variables")

## Estimate the knot states

Levenberg–Marquardt minimizes the sum of the WNOA motion costs and squared-range measurement costs. Because the problem is nonlinear, the result depends on the initial estimate; the perturbed knot states above provide a nearby starting trajectory.

In [ ]:
result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()


def knot_values(values):
    positions = np.array([
        np.asarray(gu.atAugmentedPoint3(state.pose, values).vector()).reshape(3)
        for state in states
    ])
    velocities = np.array([
        np.asarray(values.atVector(state.velocity)).reshape(3)
        for state in states
    ])
    return positions, velocities


initial_positions, initial_velocities = knot_values(initial)
estimated_positions, estimated_velocities = knot_values(result)

print(f"Initial graph error: {graph.error(initial):.4f}")
print(f"Final graph error:   {graph.error(result):.4f}")
print(f"Knot position RMSE: {np.sqrt(np.mean(np.sum((estimated_positions - true_position(knot_times))**2, axis=1))):.4f} m")

## Trajectory estimate

The estimated knot positions and velocities define a continuous trajectory between the knots. For the Euclidean WNOA model, the conditional mean follows cubic Hermite interpolation. The 3D plot compares that trajectory with ground truth and the initial estimate; the landmarks are shown for spatial reference.

In [ ]:
def interpolate_positions(times, positions, velocities):
    segment = np.clip(
        np.searchsorted(knot_times, times, side="right") - 1,
        0, len(knot_times) - 2,
    )
    dt = knot_times[segment + 1] - knot_times[segment]
    u = (times - knot_times[segment]) / dt
    return (
        (2 * u**3 - 3 * u**2 + 1)[:, None] * positions[segment]
        + (u**3 - 2 * u**2 + u)[:, None] * dt[:, None] * velocities[segment]
        + (-2 * u**3 + 3 * u**2)[:, None] * positions[segment + 1]
        + (u**3 - u**2)[:, None] * dt[:, None] * velocities[segment + 1]
    )


plot_times = np.linspace(knot_times[0], knot_times[-1], 200)
true_path = true_position(plot_times)
initial_path = interpolate_positions(plot_times, initial_positions, initial_velocities)
estimated_path = interpolate_positions(plot_times, estimated_positions, estimated_velocities)

fig = go.Figure()
fig.add_scatter3d(x=true_path[:, 0], y=true_path[:, 1], z=true_path[:, 2], mode="lines", name="Ground truth", line=dict(width=5))
fig.add_scatter3d(x=initial_path[:, 0], y=initial_path[:, 1], z=initial_path[:, 2], mode="lines", name="Initial estimate", line=dict(width=3, dash="dash"))
fig.add_scatter3d(x=estimated_path[:, 0], y=estimated_path[:, 1], z=estimated_path[:, 2], mode="lines", name="WNOA estimate", line=dict(width=5))
fig.add_scatter3d(x=estimated_positions[:, 0], y=estimated_positions[:, 1], z=estimated_positions[:, 2], mode="markers", name="Estimated knots", marker=dict(size=5))
fig.add_scatter3d(x=landmarks[:, 0], y=landmarks[:, 1], z=landmarks[:, 2], mode="markers", name="Known landmarks", marker=dict(size=6, symbol="diamond"))
fig.update_layout(title="Range-only trajectory estimate", scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)", aspectmode="data"), margin=dict(l=0, r=0, b=0, t=45))
fig.show()

## Position error

The Euclidean distance between the estimated and true positions measures trajectory accuracy in meters. This plot evaluates the continuous trajectories, not just their knot positions.

In [ ]:
initial_error = np.linalg.norm(initial_path - true_path, axis=1)
estimated_error = np.linalg.norm(estimated_path - true_path, axis=1)

fig = go.Figure()
fig.add_scatter(x=plot_times, y=initial_error, mode="lines", name="Initial estimate", line=dict(dash="dash"))
fig.add_scatter(x=plot_times, y=estimated_error, mode="lines", name="WNOA estimate")
fig.update_layout(title="Trajectory position error", xaxis_title="Time (s)", yaxis_title="Position error (m)")
fig.show()

print(f"Mean position error, initial: {initial_error.mean():.4f} m")
print(f"Mean position error, WNOA:    {estimated_error.mean():.4f} m")

## Quadratic lifting and semidefinite relaxation

A squared-range residual is quadratic in the position, so each position can be written with the lifted vector $X=[h, x, u]$, where $x$ holds the coordinates, $u=x^\top x$, and $h=1$. The lift is exact when $x^\top x = hu$ and $h^2=1$. In this form the squared-range cost is quadratic in $X$, and the WNOA motion cost is already quadratic in the knot positions and velocities, which stay as ordinary 3D vectors. Calling `gtsam.QcqpProblem(graph)` performs this conversion for the whole factor graph without approximation.

An interpolated position $x(\tau)$, a linear combination of the surrounding knot positions and velocities, is not a state, so its squared norm cannot be read from a knot variable. For each measurement time the conversion therefore introduces an auxiliary `AugmentedPoint3` with its own lift constraints, linked to the surrounding knot positions and velocities by a linear equality. Each measurement time uses one auxiliary point, shared by all ranges taken at that time. The quadratic constraints alone allow $h=\pm 1$ separately for every point, which would flip the sign of the coordinates in these linear links, so we also fix $h=1$.

The semidefinite relaxation replaces the rank-one matrix $[1;y][1;y]^\top$, built from all lifted variables $y$, with a positive semidefinite matrix whose first entry is 1. Its optimal cost is a lower bound on the QCQP cost. If a feasible point of the QCQP attains that lower bound, the point is a global minimizer and the relaxation is tight for this instance. A numerical solver only reaches the bound up to its tolerance, so the bound is meaningful only when the solver reports an optimal status, and a point that matches it to within that tolerance is optimal only to the same accuracy. If the gap is clearly nonzero, either the relaxation is not tight or the solver did not converge; the bound then only limits how far from optimal any solution can be. Solving the relaxation requires MOSEK, which GTSAM uses through `MosekMonolithicSDP` when GTSAM is configured with `GTSAM_WITH_MOSEK=ON`.

## Sparse intermediate-time measurements

To make the problem sparse, we keep all four ranges at the first knot but only two ranges, to alternating landmarks, at each intermediate time. This leaves 16 of the 28 measurements. Levenberg–Marquardt solves the sparse nonlinear problem first so the relaxation has a reference.

In [ ]:
def sparse_ranges(index, landmark):
    return index == 0 or landmark in (index % 4, (index + 1) % 4)


sparse_graph = build_graph(sparse_ranges)
sparse_result = gtsam.LevenbergMarquardtOptimizer(sparse_graph, initial).optimize()

# One auxiliary lifted point per interpolated measurement time.
interpolated_keys = [gtsam.symbol("p", 100 + i) for i in range(1, len(measurement_times))]
position_keys = [state.pose for state in states] + interpolated_keys

problem = gtsam.QcqpProblem(sparse_graph)
fix_h = np.array([[1.0, 0.0, 0.0, 0.0, 0.0]])
for key in position_keys:
    problem.addConstraint(gtsam.LinearConstraint.Equal(
        gtsam.JacobianFactor(key, fix_h, np.ones(1), gtsam.noiseModel.Unit.Create(1))))

print(f"Sparse graph: {sparse_graph.size() - len(states) + 1} range factors")
print(f"Nonlinear error after Levenberg–Marquardt: {sparse_graph.error(sparse_result):.4f}")
print(f"QCQP (cost, equality, inequality) dimensions: {problem.dim()}")

## Solve the relaxation

MOSEK solves the relaxation and returns one lifted column per variable. The relaxation's objective is a lower bound on the QCQP cost, and, for the fixed-noise interpolation factors used here, the QCQP is the original nonlinear problem rewritten exactly. Any set of knot states is feasible for the nonlinear problem, so its nonlinear cost is an upper bound. We evaluate that cost at the knot states read from the SDP solution and also keep the Levenberg–Marquardt cost. A gap near zero, within the solver's accuracy, means the best upper bound is optimal to that accuracy and the relaxation is tight for this instance. A clearly positive gap means the relaxation is not tight or the solver did not converge. Because the solver stops at a finite tolerance, the gap can be slightly negative; a magnitude near that tolerance counts as zero, and a clearly negative gap indicates a problem with the solve or the model. The cell below skips the solve with an explanation if this GTSAM build has no MOSEK support or no usable license.

In [ ]:
sdp = None
if not hasattr(gtsam, "MosekMonolithicSDP"):
    print("This GTSAM build has no MOSEK support (configure GTSAM with GTSAM_WITH_MOSEK=ON); skipping the SDP.")
else:
    try:
        sdp = gtsam.MosekMonolithicSDP(problem)
        if not sdp.solve():
            raise RuntimeError(sdp.problemStatus())
    except RuntimeError as error:
        print(f"MOSEK could not solve the relaxation: {error}")
        sdp = None

In [ ]:
def knot_errors(values):
    positions, _ = knot_values(values)
    return np.linalg.norm(positions - true_position(knot_times), axis=1)


if sdp is not None:
    qcqp_values = sdp.qcqpValues()
    sdp_values = gtsam.Values()  # knot states in the original (unlifted) variables
    for state in states:
        sdp_values.insert(state.velocity, qcqp_values.atMatrix(state.velocity)[:, 0])
        gu.insertAugmentedPoint3(
            state.pose, gu.fromQcqpValue3(qcqp_values.atMatrix(state.pose)), sdp_values)

    lower_bound = sdp.objectiveValue()
    sdp_cost = sparse_graph.error(sdp_values)
    lm_cost = sparse_graph.error(sparse_result)
    _, violation, _ = problem.evaluate(qcqp_values)
    print(f"MOSEK problem status:                  {sdp.problemStatus()}")
    print(f"SDP lower bound:                      {lower_bound:.6f}")
    print(f"Nonlinear cost at the SDP knot states: {sdp_cost:.6f}")
    print(f"Nonlinear cost, Levenberg–Marquardt:   {lm_cost:.6f}")
    print(f"Relative gap to the best upper bound:  {(min(sdp_cost, lm_cost) - lower_bound) / lower_bound:.2e}")
    print(f"Constraint violation of the lifted SDP solution: {violation:.2e}")

    fig = go.Figure()
    for name, values in (("Initial estimate", initial), ("Levenberg–Marquardt", sparse_result), ("SDP relaxation", sdp_values)):
        fig.add_bar(x=[f"Knot {i}" for i in range(len(states))], y=knot_errors(values), name=name)
    fig.update_layout(title="Knot position error, sparse measurements", yaxis_title="Position error (m)", barmode="group")
    fig.show()

## Rank diagnostics

For each variable, the eigenvalue ratio (EVR) divides the two largest eigenvalues of its block of the relaxed matrix, $[1\ y^\top; y\ Y]$. A very large ratio means the block is numerically rank one, so the relaxation produced an exact lifted point for that variable. The constraint violation printed above measures how far the lifted SDP solution is from satisfying the lift constraints and the interpolation links; a block that is exactly rank one satisfies them, but a small violation does not by itself prove that a block is rank one. These diagnostics describe the solution returned by the solver, not every optimal solution, because an optimal relaxation can have several solutions of different rank.

A knot position that has no direct range measurement enters the costs only through its coordinates $x$ in the motion and interpolation terms, so nothing in the objective constrains the second moments involving its $u$ entry. Its ratio can therefore be small even when its cost is unaffected. For such blocks, the gap between the lower bound and the upper bound is the more reliable indicator of tightness than the ratio.

In [ ]:
if sdp is not None:
    names = [gtsam.Symbol(key).string() for key in sdp.orderedKeys()]
    ratios = sdp.variableEVRs()
    fig = go.Figure(go.Bar(x=names, y=ratios))
    fig.update_layout(title="Eigenvalue ratio per variable", yaxis_title="EVR", yaxis_type="log")
    fig.show()
    for name, ratio in zip(names, ratios):
        print(f"{name:>5}: EVR {ratio:.2e}")

The range observations constrain the trajectory at their actual measurement times, even though only the knot states are estimated. The WNOA prior couples neighboring positions and velocities, giving a smooth trajectory between observations. Unlike single-point range localization, this graph estimates **a continuous path** from asynchronous range measurements.